In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time
import csv
from bs4 import BeautifulSoup


options = webdriver.ChromeOptions()

options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://www.jumia.co.ke/flash-sales/"
driver.get(url)


time.sleep(5)


driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
time.sleep(2)
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

soup = BeautifulSoup(driver.page_source, 'html.parser')
products = soup.select('article.prd._fb._p.col.c-prd')

csv_file = 'jumia_flash_sales.csv'

with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Product Name', 'Brand', 'New Price', 'Old Price', 'Discount', 'Items Left', 'Ratings'])

    for product in products:
        
        name = product.select_one('.name').get_text(strip=True) if product.select_one('.name') else "N/A"
        brand = product.get('data-brand', 'N/A')
        new_price = product.select_one('.prc').get_text(strip=True) if product.select_one('.prc') else "N/A"
        old_price = product.select_one('.old').get_text(strip=True) if product.select_one('.old') else "N/A"
        discount = product.select_one('.bdg._dsct').get_text(strip=True) if product.select_one('.bdg._dsct') else "0%"
        
        
        items_left_el = product.select_one('.stk .rev')
        items_left = items_left_el.get_text(strip=True) if items_left_el else "Not specified"
        
       
        rating_el = product.select_one('.stars._s')
        rating = rating_el.get_text(strip=True) if rating_el else "No ratings"

        
        writer.writerow([name, brand, new_price, old_price, discount, items_left, rating])

print(f"Successfully scraped {len(products)} products to {csv_file}")

Successfully scraped 40 products to jumia_flash_sales.csv
